# Faz 1 advanced retrieval smoke

Output labels used throughout this notebook, per
`docs/planning/ADVANCED_RETRIEVAL_FINAL_PLAN_v2.1.md` Sec.8:

- **plumbing_smoke**: proves the code path runs end-to-end against a real backend.
  Not a quality claim.
- **vector_math_latency**: real wall-clock timing of the actual calls made here, on
  this machine (CPU-only, non-representative GPU). Not a production latency claim.
- **offline_quality**: research-plane quality signal only, never presented as a
  production ClickHouse result.

This notebook does not replace `service/tests/test_advret_*.py` -- those are the
authoritative, CI-runnable regression tests for the Phase -1 fixes. This notebook is a
human-readable walkthrough of the same product-plane code (`service/app/**`), run against
whatever live backend is reachable, with everything it could not verify honestly labeled
`not_run` rather than guessed.

In [1]:
import os
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SERVICE_ROOT = REPO_ROOT / "service"
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8143")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "")
os.environ.setdefault("CLICKHOUSE_DB", "uav_search")

from app.db import clickhouse

reachable = clickhouse.health()
print("clickhouse.health():", reachable)
print("status:", "plumbing_smoke will run against a real backend" if reachable else "NOT_RUN -- no live ClickHouse reachable from this kernel")

clickhouse.health(): True
status: plumbing_smoke will run against a real backend


In [2]:
import numpy as np

DATASET_ID = "advret_notebook09_smoke"
DIMENSION = 512
N_ROWS = 40

def unit_vector(seed: int) -> list[float]:
    rng = np.random.default_rng(seed)
    v = rng.standard_normal(DIMENSION).astype(np.float32)
    return (v / np.linalg.norm(v)).tolist()

rows = [
    {
        "segment_id": f"nb09_{i:03d}", "dataset_id": DATASET_ID, "video_id": "nb09_video",
        "t_start": float(i), "t_end": float(i + 1), "altitude_m": 10.0, "velocity_mps": 1.0,
        "gimbal_pitch": 0.0, "person_count": i % 4, "vehicle_count": i % 2, "is_night": 0,
        "embedding": unit_vector(i),
    }
    for i in range(N_ROWS)
]

result_summary = {"status": "NOT_RUN", "reason": "no live ClickHouse reachable"}
if reachable:
    clickhouse.replace_vectors(DATASET_ID, DIMENSION, rows)
    result_summary = {"status": "plumbing_smoke", "rows_written": N_ROWS}
print(result_summary)

{'status': 'plumbing_smoke', 'rows_written': 40}


In [3]:
# vector_math_latency: real timing of search_vectors() against the live corpus written
# above, both the plain path and the adaptive-MRL-style two-stage path (base_dim/top_n
# then dimension/top_k over the stage-1 candidate ids). This is CPU/this-machine timing,
# not a production latency claim -- see artifacts/advanced_retrieval/backends/ for the
# real backend benchmark.

timings = {}
if reachable:
    query_vector = unit_vector(999)

    started = time.perf_counter()
    plain_rows, plain_diag = clickhouse.search_vectors(
        DATASET_ID, DIMENSION, query_vector, top_k=5, strategy="exact", candidate_ids=None, diagnose=True,
    )
    timings["single_stage_ms"] = round((time.perf_counter() - started) * 1000, 3)

    started = time.perf_counter()
    stage1_rows, stage1_diag = clickhouse.search_vectors(
        DATASET_ID, DIMENSION, query_vector, top_k=15, strategy="exact", candidate_ids=None, diagnose=True,
    )
    stage1_ids = [row["segment_id"] for row in stage1_rows]
    stage2_rows, stage2_diag = clickhouse.search_vectors(
        DATASET_ID, DIMENSION, query_vector, top_k=5, strategy="exact", candidate_ids=stage1_ids, diagnose=True,
    )
    timings["two_stage_ms"] = round((time.perf_counter() - started) * 1000, 3)

    print("plumbing_smoke: single-stage rows =", len(plain_rows), "| two-stage rows =", len(stage2_rows))
    print("plain diagnostics.candidate_count (== filtered corpus, no candidate restriction):", plain_diag["candidate_count"])
    print("stage2 diagnostics.candidate_count (== stage-1 output size, the Phase -1.1 fix):", stage2_diag["candidate_count"], "vs stage1 output size", len(stage1_ids))
    assert stage2_diag["candidate_count"] == len(stage1_ids), "regression: stage-2 candidate_count should equal the stage-1 restriction size, not the full corpus"
    print("vector_math_latency:", timings)
else:
    print("NOT_RUN")

plumbing_smoke: single-stage rows = 5 | two-stage rows = 5
plain diagnostics.candidate_count (== filtered corpus, no candidate restriction): 40
stage2 diagnostics.candidate_count (== stage-1 output size, the Phase -1.1 fix): 15 vs stage1 output size 15
vector_math_latency: {'single_stage_ms': 55.877, 'two_stage_ms': 272.834}


In [4]:
# cleanup -- this notebook only ever touches its own dataset_id, never auair/capera or
# any other real institution data.
if reachable:
    clickhouse.replace_vectors(DATASET_ID, DIMENSION, [])
    remaining = clickhouse.table_count(DATASET_ID, DIMENSION)
    print("cleanup complete, remaining rows for", DATASET_ID, "=", remaining)
    assert remaining == 0

cleanup complete, remaining rows for advret_notebook09_smoke = 0
